In [ ]:
from pathlib import Path
import pandas as pd
from catboost import CatBoostClassifier, Pool

In [ ]:
TRAIN_PATH = Path('train.csv')
TEST_PATH = Path('test.csv')
OUTPUT_PATH = Path('submission_catboost_baseline.csv')

ID_COL = 'claim_id'
TARGET = 'is_valid'
THRESHOLD = 0.777

DROP_COLS = [
    TARGET,
    ID_COL,
    'id_content_owner',
    'id_content',
    'first_event_time',
    'content_registered_time',
]

In [ ]:
def prepare_features(train: pd.DataFrame, test: pd.DataFrame):
    y = train[TARGET].astype(int)
    test_ids = test[ID_COL].copy()

    train_features = train.copy()
    test_features = test.copy()

    train_features = train_features.drop(columns=[c for c in DROP_COLS if c in train_features.columns])
    test_features = test_features.drop(columns=[c for c in DROP_COLS if c in test_features.columns])

    cat_cols = train_features.select_dtypes(include=['object', 'string']).columns.tolist()
    for col in cat_cols:
        train_features[col] = train_features[col].fillna('missing').astype(str)
        test_features[col] = test_features[col].fillna('missing').astype(str)

    cat_features = [train_features.columns.get_loc(col) for col in cat_cols]
    return train_features, test_features, y, test_ids, cat_features

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)

X, X_test, y, test_ids, cat_features = prepare_features(train, test)

print(X.shape)
print(X_test.shape)
print(round(y.mean(), 4))

In [ ]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
)


model.fit(Pool(X, y, cat_features=cat_features))

In [ ]:
test_pred = model.predict_proba(X_test)[:, 1]
test_label = (test_pred >= THRESHOLD).astype(int)
submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_label})
submission.to_csv(OUTPUT_PATH, index=False)
submission.head(10)

In [ ]:
print(submission.shape)
print(submission[TARGET].value_counts(normalize=True))